In [1]:
# --- standard library imports ---
import numpy as np
import pandas as pd
from pathlib import Path

from ase import Atoms
from ase.io import read, Trajectory
from ase.visualize import view
from pymatgen.io.ase import AseAtomsAdaptor

# --- IMPORTS FROM THIS PROJECT ---
from gnome_dreams_oer_screening import OER_catalyst_study
from gnome_dreams_oer_screening import RelationalFrame 
from gnome_dreams_oer_screening import Explog
from gnome_dreams_oer_screening.vasp.Job_handler import Job_handler

In [2]:
explog = Explog()

TypeError: Explog.__init__() missing 1 required positional argument: 'queue_adapter'

In [ ]:
# Test structures, to hand to the study object:
ZrO2_m = read('ZrO2_m.cif')
TiO2_r = read('TiO2_rutile.cif')
test_cands = Trajectory('test_cands.traj')

atoms = test_cands[18]
S = AseAtomsAdaptor.get_structure(atoms)

In [ ]:
# Creation of ExpLog, to be used thuroughout:
EXPLOG = RelationalFrame(root="candidates")

study_df = pd.DataFrame(columns=["candidate_id", "reason_or_hypothesis",
 "notes", "study_obj", "OHDone", "idealOverPotential"])

EXPLOG.add_table("candidates", study_df, pk="candidate_id")

process_df = pd.DataFrame(columns=["process_id", "candidate_id", 
            "job_type", "slurmID", "status", "VASP_dir", "processNote"])
EXPLOG.add_table("processes", process_df, pk="process_id")

EXPLOG.candidates.add_relation(
    name="processes",
    child_table="processes",
    fk_in_child="candidate_id",
)

<Table candidates rows=0 pk=candidate_id>

In [ ]:
# Creation
JOB_HANDLER = Job_handler(EXPLOG, mode="test", 
    exlib_cache_dir = 'TEMP',
    vasp_calculation_dir = Path("TEMP/vasp_calculations/"),
    overwrite_existing_vasp_dirs=True
    )

# NOTE: the job_submitter needs both the candidates and the progress df
# to be in the ExpLog, to fucntion.

# modes: 
# "test" - ...
# "production_test" - ...
# "production" - ...

### From this point the agent would start using tools, operating on the log etc.

In [ ]:
# Dummy setup of frist row (for testing):
study = OER_catalyst_study(atoms)
cant_id = "07286qnfd"

EXPLOG.candidates.add_row(
    {"candidate_id": cant_id,
     "reason_or_hypothesis": "Ir sites should be great", 
     "notes": "this is a great candidate", 
     "study_obj": study},
     allow_update=True,
)

EXPLOG.candidates.df

,candidate_id,reason_or_hypothesis,notes,study_obj,OHDone,idealOverPotential
0,07286qnfd,Ir sites should be great,this is a great candidate,<gnome_dreams_oer_screening.oer.oer_study.OER_...,<NA>,<NA>


Having entered the candidate in the log, now one would submit a bulk job:

In [ ]:
JOB_HANDLER.enter_in_log(
    candidate_id = cant_id,
    calculation_type = "bulk_relaxation"
)

In [ ]:
EXPLOG.processes.df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,10,07286qnfd,bulk_relaxation,<NA>,un-submitted,TEMP\vasp_calculations\07286qnfd_bulk_relaxati...,<NA>


In [ ]:
JOB_HANDLER.submit_vasp_calculation(
    candidate_id = cant_id, process_id = str(10), resource = 'xeon24el8'
    )

EXPLOG.processes.df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,10,07286qnfd,bulk_relaxation,<NA>,submitted,TEMP\vasp_calculations\07286qnfd_bulk_relaxati...,<NA>


In [ ]:
cant_id = "sdaf32rdf"
EXPLOG.candidates.add_row(
    {"candidate_id": cant_id,
     "reason_or_hypothesis": "Ir sites should be great", 
     "notes": "this is a great candidate", 
     "study_obj": study},
     allow_update=True,
)
JOB_HANDLER.enter_in_log(
    candidate_id = cant_id,
    calculation_type = "bulk_relaxation"
)


In [ ]:
JOB_HANDLER.update_log().df

,process_id,candidate_id,job_type,slurmID,status,VASP_dir,processNote
0,10,07286qnfd,bulk_relaxation,<NA>,submitted,TEMP\vasp_calculations\07286qnfd_bulk_relaxati...,<NA>
1,20,sdaf32rdf,bulk_relaxation,<NA>,un-submitted,TEMP\vasp_calculations\sdaf32rdf_bulk_relaxati...,<NA>
